In [ ]:
def patch_environment():
    """Patches missing triton.ops, language attributes, and handles CPU/GPU fallbacks for Kaggle and Colab."""
    import os
    import sys
    import types
    import importlib.machinery

    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 1. Patch Triton mock safely
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__spec__ = importlib.machinery.ModuleSpec("triton", None)
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    # Ensure triton.language exists to satisfy transformers.modeling_utils import checks
    if not hasattr(m_triton, "language"):
        m_lang = types.ModuleType("triton.language")
        m_lang.__spec__ = importlib.machinery.ModuleSpec("triton.language", None)
        m_lang.constexpr = lambda x: x
        setattr(m_triton, "language", m_lang)
        sys.modules["triton.language"] = m_lang

    if not hasattr(m_triton, "cdiv"):
        setattr(m_triton, "cdiv", lambda a, b: (a + b - 1) // b)
    if not hasattr(m_triton, "jit"):
        setattr(m_triton, "jit", lambda *a, **k: (lambda f: f))

    m_ops = types.ModuleType("triton.ops")
    m_ops.__spec__ = importlib.machinery.ModuleSpec("triton.ops", None)
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.__spec__ = importlib.machinery.ModuleSpec("triton.ops.matmul_perf_model", None)
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 2. Bulletproof fallback for bitsandbytes
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes", None)
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")
        m_bnb_nn.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn", None)

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb_nn.modules.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn.modules", None)
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.47.0"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️ CẢNH BÁO: BẠN CHƯA BẬT GPU!')
    print('👉 Vui lòng vào menu Runtime -> Change runtime type -> Chọn T4 GPU (hoặc Dual T4 trên Kaggle) -> Nhấn Save và chạy lại từ đầu.')


In [ ]:
# Clone mã nguồn và cài đặt thư viện cần thiết
!rm -rf Research-assist
!git clone https://github.com/nguyentatmanh/Research-assist.git
%cd Research-assist
!git checkout upgrade 2>/dev/null || true

!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    datasets==3.0.2 \
    accelerate>=1.1.0 \
    safetensors \
    scikit-learn scipy einops ipywidgets

print("✅ Đã chuẩn bị xong môi trường Colab!")


In [ ]:
# ==============================================================================
# BƯỚC: HUẤN LUYỆN AGENT 2 LAYOUTLMv3 (DOCUMENT READER)
# ==============================================================================
import os, sys, torch

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# Khởi chạy huấn luyện Agent 2 Document Reader trên GPU T4 của Colab
!{sys.executable} -u training_lab/agent2_reader/train_agent2_t4x2.py --epochs 3


In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

zip_path = "Agent2_LayoutLMv3_DocLayNet.zip"
weights_dir = Path("training_lab/agent2_reader/weights")

if not weights_dir.exists():
    print("⚠️ Chưa tìm thấy thư mục weights. Vui lòng đảm bảo Bước huấn luyện Agent 2 hoàn tất thành công.")
else:
    print("⏳ Đang chuẩn bị và tối ưu dung lượng...")
    # Xóa checkpoint phụ
    !rm -rf training_lab/agent2_reader/weights/checkpoints
    !zip -r -q {zip_path} training_lab/agent2_reader/weights/
    print(f"\n🎉 Đã đóng gói thành công: {zip_path}")
    print("👉 Bấm vào nút màu xanh bên dưới để tải file về máy:")

    download_btn = widgets.Button(
        description="📥 Click để tải Agent2_LayoutLMv3_DocLayNet.zip",
        button_style="success",
        layout=widgets.Layout(width="420px", height="50px")
    )
    def on_click_download(b):
        if os.path.exists(zip_path):
            print("🚀 Trình duyệt đang bắt đầu tải file về máy tính...")
            files.download(zip_path)
        else:
            print(f"❌ Không tìm thấy file {zip_path}.")
    download_btn.on_click(on_click_download)
    display(download_btn)


In [ ]:
# ==============================================================================
# BƯỚC: HUẤN LUYỆN AGENT 4 FACT-CHECKER (NLI VERIFIER)
# ==============================================================================
import os, sys, torch

print("=" * 80)
print("🛡️ HUẤN LUYỆN AGENT 4 FACT-CHECKER (BATCH=4 + GRADIENT CHECKPOINTING)")
print("=" * 80)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# Khởi chạy huấn luyện 3 Epochs
!{sys.executable} -u training_lab/agent4_fact_checker/train_agent4_t4x2.py --epochs 3 --batch-size 4


In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

zip_path = "Agent4_FactChecker_NLI.zip"
weights_dir = Path("training_lab/agent4_fact_checker/weights")

if not weights_dir.exists():
    print("⚠️ Chưa tìm thấy thư mục weights của Agent 4. Vui lòng chạy bước huấn luyện trước.")
else:
    print("⏳ Đang dọn dẹp và tối ưu hóa dung lượng Agent 4...")
    !rm -rf training_lab/agent4_fact_checker/weights/checkpoints
    !zip -r -q {zip_path} training_lab/agent4_fact_checker/weights/
    print(f"\n🎉 Đã đóng gói thành công: {zip_path}")
    print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

    download_btn = widgets.Button(
        description="📥 Click để tải Agent4_FactChecker_NLI.zip",
        button_style="success",
        layout=widgets.Layout(width="420px", height="50px")
    )
    def on_click_download(b):
        if os.path.exists(zip_path):
            print("🚀 Trình duyệt đang bắt đầu tải Agent 4 về máy tính...")
            files.download(zip_path)
        else:
            print(f"❌ Không tìm thấy file {zip_path}.")
    download_btn.on_click(on_click_download)
    display(download_btn)


In [ ]:
import os, sys, torch
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("=" * 80)
print("🎉 HUẤN LUYỆN & ĐÓNG GÓI AGENT 1: ROUTER & PLANNER")
print("=" * 80)

weights_dir = Path("training_lab/agent1_retrieval/weights")
zip_path = "Agent1_Router_Planner.zip"

if not weights_dir.exists() or not list(weights_dir.glob("*")):
    print("⏳ Đang huấn luyện Agent 1 (Router & Planner) trên tập dữ liệu chuẩn (~1-2 phút)...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    !{sys.executable} -u training_lab/agent1_retrieval/train_agent1_t4x2.py --epochs 3

if not weights_dir.exists():
    print("⚠️ Huấn luyện chưa tạo ra thư mục weights. Vui lòng kiểm tra lại log huấn luyện.")
else:
    # Dọn dẹp checkpoints phụ trung gian
    !rm -rf training_lab/agent1_retrieval/weights/checkpoints
    !zip -r -q {zip_path} training_lab/agent1_retrieval/weights/

    print(f"\n🎉 Đã đóng gói thành công: {zip_path}")
    print("👉 Bấm nút màu xanh bên dưới để tải về máy tính:")

    download_btn = widgets.Button(
        description="📥 Click để tải Agent1_Router_Planner.zip",
        button_style="success",
        layout=widgets.Layout(width="420px", height="50px")
    )
    def on_click_download(b):
        if os.path.exists(zip_path):
            print("🚀 Đang bắt đầu tải Agent 1 về máy tính...")
            files.download(zip_path)
        else:
            print(f"❌ Không tìm thấy file {zip_path}.")
    download_btn.on_click(on_click_download)
    display(download_btn)
